In [ ]:
# !pip install pandas numpy faiss-cpu openai ipywidgets openpyxl

In [ ]:
import os
import re
from pathlib import Path
import pandas as pd

# === GLOBAL CONFIG ===
os.environ["TOKENIZERS_PARALLELISM"] = "false"
DATA_DIR = Path(".")
INDEX_DIR = DATA_DIR / "index"
EMB_MODEL_NAME = "text-embedding-3-small"  # embedding model for retrieval
OPENAI_MODEL = "gpt-4.1-mini"  # model for answer generation
DEFAULT_TOP_K = 6
API_KEY_ENV = "OPENAI_API_KEY"
API_KEY_PATH = DATA_DIR / "apikey.env"

# === FILE PATHS ===
INPUT_CSVS = sorted(DATA_DIR.glob('BYD_FGD*_transcript_turns.csv'))
OUTPUT_CSV_CLEAN = "API_BYD_FGD_all_transcript_turns_cleaned.csv"

# Preview settings for console output
CLEAN_SAMPLE_ROWS = 10

assert INPUT_CSVS, "No input CSVs found matching pattern BYD_FGD*_transcript_turns.csv"

# === LOAD DATA FROM CSVs ===
frames = []
for csv_path in INPUT_CSVS:
    print(f"Loading {csv_path.name} ...")
    df_part = pd.read_csv(csv_path)
    df_part["source_file"] = csv_path.name
    frames.append(df_part)

df = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(INPUT_CSVS)} files.")
print("Columns found:", df.columns.tolist())
print("Number of rows before cleaning:", len(df))

# Make sure we actually have an 'utterance' column
assert "utterance" in df.columns, "Expected an 'utterance' column in the input CSVs."


# ---------- 1. Helper: remove filler words ----------

FILLER_PATTERNS = [
    r"\buh\b",
    r"\bum\b",
    r"\berm\b",
    r"\ber\b",
    r"\burr\b",
    r"\bah\b",
    r"\buhm\b",
    r"\bmmm\b",
    r"\byou know\b",
    r"\bkind of\b",
    r"\bsort of\b",
    r"\bi mean\b",
    r"\bi guess\b",
    r"\bright\?\b",
    r"\bok,\b",
    r"\bok.\b",
    r"\boh,\b",
    r"\bso,\b",
    r"\bok\b",
    r"\bokay\b",
    r"\bwell,\b",
]

FILLER_REGEXES = [re.compile(pat, flags=re.IGNORECASE) for pat in FILLER_PATTERNS]


def remove_filler_words(text: str) -> str:
    if not isinstance(text, str):
        return text
    new_text = text
    for regex in FILLER_REGEXES:
        new_text = regex.sub(" ", new_text)
    return new_text


# ---------- 2. Helper: collapse repeated words ----------

def collapse_repeated_words(text: str) -> str:
    """
    Turns:
        "OK OK OK, I think..." -> "OK, I think..."
        "ya ya" -> "ya"
    Only for directly repeated words.
    """
    if not isinstance(text, str):
        return text

    pattern = re.compile(r"\b(\w+)(\s+\1\b)+", flags=re.IGNORECASE)
    return pattern.sub(r"\1", text)


# ---------- 3. Master cleaning function for utterances ----------

def clean_utterance(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.strip()
    text = remove_filler_words(text)
    text = collapse_repeated_words(text)


    # Punctuation tidy: drop leading stray punct (even after quotes), trim isolated commas/periods, collapse repeats, drop trailing lone period
    text = re.sub(r"^[^A-Za-z0-9]+(?=[A-Za-z0-9])", "", text)
    text = re.sub(r"^[\"\']*[.,!?]+\s*", "", text)
    text = re.sub(r"\s+[,\.]{1,}\s+", " ", text)
    text = re.sub(r"([,\.\.!?…]){2,}", r"\1", text)
    text = re.sub(r"\s*\.$", "", text)  # drop trailing lone period like "word ."

    text = re.sub(r"\s+", " ", text).strip()
    return text


# ---------- 4. Apply cleaning pipeline (no row removals) ----------

# Keep original utterance for audit if not already present
if "utterance_original" not in df.columns:
    df["utterance_original"] = df["utterance"]

# Clean the utterance text in place
df["utterance"] = df["utterance"].apply(clean_utterance)

# Turn IDs stay sequential by session
if "session_id" in df.columns:
    if "turn_id" in df.columns:
        df = df.sort_values(["session_id", "turn_id"]).reset_index(drop=True)
    else:
        df = df.sort_values(["session_id"]).reset_index(drop=True)
    df["turn_id"] = df.groupby("session_id").cumcount() + 1
else:
    df = df.reset_index(drop=True)
    df["turn_id"] = df.index + 1

print("Number of rows after cleaning (no removals):", len(df))
print(f"\nSample of cleaned output (first {CLEAN_SAMPLE_ROWS} rows):")
print(df.head(CLEAN_SAMPLE_ROWS))

# ---------- 5. Save cleaned data ----------

df.to_csv(OUTPUT_CSV_CLEAN, index=False, encoding="utf-8-sig")
print(f"\nCleaned data saved to: {OUTPUT_CSV_CLEAN}")


In [ ]:
import pandas as pd
import textwrap

INPUT_CSV_CLEAN = OUTPUT_CSV_CLEAN

df = pd.read_csv(DATA_DIR / INPUT_CSV_CLEAN)
print(df.columns)
print(len(df))

# Basic sanity: assume we have columns: session_id, turn_id, speaker_name, speaker_role, utterance
df = df.sort_values(["session_id", "turn_id"]).reset_index(drop=True)


# ---- Chunking helper ----
def make_chunks(df, max_chars=1000, overlap_chars=150):
    """
    Greedy chunking by turn, keeping turns for context.
    max_chars is per chunk; overlap_chars is how much text to repeat between chunks.
    """
    chunks = []
    current = ""
    current_meta = []
    current_session = None

    for _, row in df.iterrows():
        row_session = row["session_id"]
        # If session changes, flush current chunk without overlap
        if current_session is None:
            current_session = row_session
        elif row_session != current_session:
            if current:
                chunks.append({
                    "session_id": current_meta[0]["session_id"],
                    "text": current.strip(),
                    "turn_start": current_meta[0]["turn_id"],
                    "turn_end": current_meta[-1]["turn_id"],
                })
            current = ""
            current_meta = []
            current_session = row_session

        text = str(row["utterance"])
        turn_info = (
            f'{row["speaker_name"]} ({row["speaker_role"]}) '
            f'[turn {row["turn_id"]}]: {text}\n'
        )

        if len(current) + len(turn_info) > max_chars and current:
            # save current chunk
            chunks.append({
                "session_id": current_meta[0]["session_id"],
                "text": current.strip(),
                "turn_start": current_meta[0]["turn_id"],
                "turn_end": current_meta[-1]["turn_id"],
            })
            # start new chunk with overlap
            overlap = current[-overlap_chars:]
            current = overlap + "\n" + turn_info
            current_meta = [row]
        else:
            current += turn_info
            current_meta.append(row)

    if current:
        chunks.append({
            "session_id": current_meta[0]["session_id"],
            "text": current.strip(),
            "turn_start": current_meta[0]["turn_id"],
            "turn_end": current_meta[-1]["turn_id"],
        })

    return pd.DataFrame(chunks)


chunks_df = make_chunks(df, max_chars=1000, overlap_chars=200)
# Stable chunk id for alignment with FAISS
chunks_df["chunk_id"] = chunks_df.apply(
    lambda r: f"s{r['session_id']}_t{r['turn_start']}-{r['turn_end']}", axis=1
)
print(chunks_df.head())
print("Num chunks:", len(chunks_df))


In [ ]:

# Export chunks to Excel
# Requires chunking cell to have run (chunks_df in memory)
if 'chunks_df' not in globals():
    raise RuntimeError("Run the chunking cell first to create chunks_df.")

export_path = "API_chunks_export.xlsx"
chunks_df.to_excel(export_path, index=False)
print(f"Saved chunks to: {export_path}")


In [ ]:
# Shared prompt builder

def build_prompt(context, question):
    return (
        "You are a research analyst reviewing a focus group transcript.\n"
        "Use only the context to answer the question.\n"
        "Answer in 2–3 sentences and cite specific details from the context.\n"
        "If the answer is not in the context, say 'Not enough context.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


In [ ]:
# Shared helpers
import os


def _load_openai_key():
    if os.getenv(API_KEY_ENV):
        return
    if API_KEY_PATH.exists():
        for line in API_KEY_PATH.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            if key.strip() == API_KEY_ENV and value.strip():
                os.environ[API_KEY_ENV] = value.strip().strip('"').strip("'")
                break


def ensure_llm():
    """Initialize the OpenAI client if it is not already set."""
    global _openai_client
    if '_openai_client' in globals():
        return
    _load_openai_key()
    api_key = os.getenv(API_KEY_ENV)
    if not api_key:
        raise RuntimeError("Set OPENAI_API_KEY in your environment or apikey.env.")
    from openai import OpenAI
    _openai_client = OpenAI(api_key=api_key)


def embed_texts(texts, batch_size=128):
    ensure_llm()
    if not texts:
        return []
    safe_texts = [t if isinstance(t, str) else "" for t in texts]
    embeddings = []
    for i in range(0, len(safe_texts), batch_size):
        batch = safe_texts[i:i + batch_size]
        response = _openai_client.embeddings.create(
            model=EMB_MODEL_NAME,
            input=batch,
        )
        batch_embeddings = [None] * len(batch)
        for item in response.data:
            batch_embeddings[item.index] = item.embedding
        embeddings.extend(batch_embeddings)
    return embeddings


def ensure_retrieval():
    """Ensure search_chunks is defined; auto-load from saved index if possible."""
    global search_chunks, chunks_df
    if 'search_chunks' in globals():
        return
    emb_name_path = INDEX_DIR / 'API_fgd_all_emb_model_name.txt'
    faiss_path = INDEX_DIR / 'API_fgd_all.index'
    chunks_path = INDEX_DIR / 'API_fgd_all_chunks.pkl'
    if not (emb_name_path.exists() and faiss_path.exists() and chunks_path.exists()):
        raise RuntimeError("Run the retrieval cell first to define search_chunks.")
    import faiss
    import numpy as np
    import pandas as pd
    chunks_df = pd.read_pickle(chunks_path)
    faiss_index = faiss.read_index(str(faiss_path))

    def search_chunks(query, k=DEFAULT_TOP_K):
        q_emb = embed_texts([query])[0]
        q_vec = np.array([q_emb], dtype="float32")
        faiss.normalize_L2(q_vec)
        D, I = faiss_index.search(q_vec, k)
        results = []
        for score, idx in zip(D[0], I[0]):
            row = chunks_df.iloc[idx]
            cid = row.get('chunk_id') or f"s{row['session_id']}_t{row['turn_start']}-{row['turn_end']}"
            results.append({
                'chunk_id': cid,
                'score': float(score),
                'session_id': str(row.get('session_id', '')),
                'turn_start': int(row.get('turn_start', 0)),
                'turn_end': int(row.get('turn_end', 0)),
                'text': row.get('text', ''),
            })
        return results


def generate_answer(prompt, temperature=None, top_p=None):
    ensure_llm()
    params = {
        "model": OPENAI_MODEL,
        "input": prompt,
    }
    if temperature is not None:
        params["temperature"] = temperature
    if top_p is not None:
        params["top_p"] = top_p
    response = _openai_client.responses.create(**params)
    return response.output_text.strip()



In [ ]:
# retrieval system
import faiss
import numpy as np

texts = chunks_df["text"].fillna("").astype(str).tolist()
if not texts:
    raise RuntimeError("No text available to embed.")

embeddings = embed_texts(texts, batch_size=128)
embeddings = np.array(embeddings, dtype="float32")
faiss.normalize_L2(embeddings)

print("Embeddings shape:", embeddings.shape)

# Build FAISS index
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("Index size:", index.ntotal)

# Save everything for later reuse
INDEX_DIR.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(INDEX_DIR / "API_fgd_all.index"))
chunks_df.to_pickle(INDEX_DIR / "API_fgd_all_chunks.pkl")
with open(INDEX_DIR / "API_fgd_all_emb_model_name.txt", "w") as f:
    f.write(EMB_MODEL_NAME)


In [ ]:
# Single-method RAG QA widget (direct RAG only)
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import textwrap

ensure_retrieval()
ensure_llm()  # initializes the OpenAI client


def run_rag(question, k=DEFAULT_TOP_K):
    hits = search_chunks(question, k=k)
    context = "\n\n".join(
        f"[{h['chunk_id']}] {textwrap.shorten(h['text'], width=500, placeholder='...')}"
        for h in hits
    )
    prompt = build_prompt(context, question)
    answer = generate_answer(prompt)
    return answer, hits


question_box = widgets.Textarea(
    value="",
    description="Question",
    placeholder="Type a question",
    layout=widgets.Layout(width="100%", height="80px"),
)
run_button = widgets.Button(description="Enter", button_style="primary")
output = widgets.Output()


def on_run_clicked(_):
    q = question_box.value.strip()
    output.clear_output()
    if not q:
        with output:
            print("Enter a question to run RAG.")
        return
    try:
        ans, hits = run_rag(q)
        with output:
            display(Markdown(f"**Question:** {q}"))
            display(Markdown(f"**Answer:** {ans}"))
            if hits:
                print("Sources:")
                for h in hits:
                    snippet = textwrap.shorten(h.get('text', ''), width=180, placeholder='...')
                    print(f"- {h.get('chunk_id')}: {snippet}")
    except Exception as exc:
        with output:
            print(f"Error: {exc}")


run_button.on_click(on_run_clicked)
controls = widgets.VBox([question_box, run_button])
display(widgets.VBox([controls, output]))

# Can you summarise what this transcript is about